<a href="https://colab.research.google.com/github/Gustavo-kohler/tcc_pinn_fwi/blob/main/pinn_fwi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto de TCC: PINNs para FWI
**Autor:** Gustavo Luiz Kohler | **Dataset:** OpenFWI (FlatVel-A)

Este notebook configura o ambiente para o treinamento de Redes Neurais Informadas pela Física (PINNs).
O backend numérico utilizado é o **PyTorch**, com abstração de grafos computacionais e geometria gerenciada pelo **DeepXDE**.

### Integração com o Google Drive
Para evitar o carregamento local dos dados do FlatVel-A a cada sessão, montamos o Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Estruturação de Diretórios e Dependências
Aqui criamos a arquitetura de pastas padronizada para projetos de Machine Learning.
Em seguida, instalamos o DeepXDE e outras dependências.

In [ ]:
%cd /content/drive/MyDrive/

!mkdir -p tcc_pinn_fwi/data/raw
!mkdir -p tcc_pinn_fwi/src
!mkdir -p tcc_pinn_fwi/models

%cd tcc_pinn_fwi/

!pip install deepxde numpy matplotlib scipy

/content/drive/MyDrive
/content/drive/MyDrive/tcc_pinn_fwi


### Validação do Motor Matemático e Hardware
Imporação das bibliotécas e configuração do DeepXDE para adotar o PyTorch como backend e validamos a alocação da placa de vídeo (GPU) fornecida pela nuvem.

In [ ]:
import os
import time
import torch
import numpy as np

os.environ["DDE_BACKEND"] = "pytorch"
import deepxde as dde

print("--- DIAGNÓSTICO DO SISTEMA ---")
print(f"Backend do DeepXDE configurado para: {dde.backend.backend_name}")
print(f"Versão do PyTorch: {torch.__version__}")

if torch.cuda.is_available():
    print(f"Acelerador de Hardware ATIVO: {torch.cuda.get_device_name(0)}")
    print(f"Memória Total da GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("ALERTA: O ambiente está rodando apenas em CPU. O treinamento da PINN será extremamente lento. Ative a GPU no menu do Colab.")

--- DIAGNÓSTICO DO SISTEMA ---
Backend do DeepXDE configurado para: pytorch
Versão do PyTorch: 2.11.0+cu128
Acelerador de Hardware ATIVO: Tesla T4
Memória Total da GPU: 15.64 GB


### Pré-processamento dos Dados
Nesta etapa, isolamos o sismograma de uma única fonte emissora e utilizamos a biblioteca NumPy para mapear o espaço `(1000 tempos x 70 sensores)` em malhas de coordenadas geográficas (`np.meshgrid`).
Em seguida, "achatamos" essa malha (`reshape`) para gerar o conjunto de treinamento tabular:
* Entradas (`X`): Colunas unificadas das posições de cada medição `[x, z, t]`.
* Saídas (`y`): Coluna contendo o valor da pressão acústica medida `[u]`.

In [ ]:
data_path = '/content/drive/MyDrive/tcc_pinn_fwi/data/raw/data1.npy'

def load_sensor_data(data_path, id_sample=0, id_source=2):
  data = np.load(data_path)

  seismogram = data[id_sample, id_source, :, :]

  t_array = np.linspace(0.0, 1.0, 1000)
  x_array = np.linspace(0.0, 700.0, 70)

  T_grid, X_grid = np.meshgrid(t_array,x_array, indexing='ij')
  Z_grid = np.zeros_like(X_grid)

  X_input = np.hstack((
      X_grid.reshape(-1, 1),
      Z_grid.reshape(-1, 1),
      T_grid.reshape(-1, 1),
  ))
  y_output = seismogram.reshape(-1, 1)

  return X_input, y_output

X_train, y_train = load_sensor_data(data_path)
print("Formato X (Sensores):", X_train.shape)
print("Formato Y (Pressão):", y_train.shape)


Formato X (Sensores): (70000, 3)
Formato Y (Pressão): (70000, 1)


### Adimensionalização do Problema

As variáveis do problema ocupam faixas numéricas muito distintas: as coordenadas espaciais chegam a
$700$, o tempo vai até $1$ e a pressão registrada atinge algumas dezenas. Redes neurais com ativação
`tanh` e inicialização de Glorot pressupõem entradas próximas de $[-1, 1]$; fora dessa faixa a ativação
opera em sua região constante, onde a derivada é nula e o gradiente deixa de se propagar pelas camadas.

A adimensionalização resolve isso reescrevendo o problema em variáveis normalizadas. Escolhem-se
grandezas de referência $L$, $T$ e $P$ para comprimento, tempo e pressão, e as variáveis originais são
expressas como múltiplos delas:

$$\hat{x} = \frac{x}{L}, \qquad \hat{z} = \frac{z}{L}, \qquad \hat{t} = \frac{t}{T},
\qquad \hat{p} = \frac{p}{P}$$

Adotamos $L = 700$ m (a extensão do domínio), $T = 1$ s (a janela de gravação) e
$P = \max|p_{obs}|$. Com isso, o domínio espaço-temporal torna-se o cubo unitário e a pressão observada
fica contida em $[-1, 1]$, faixa adequada tanto para as entradas quanto para a saída da rede.

A normalização ajusta a escala das entradas e da saída, mas não a das
derivadas de alta ordem. Um campo que oscila várias vezes ao longo do domínio possui derivadas segundas
cuja magnitude, mesmo em variáveis normalizadas, permanece na ordem de $10^3$. Os termos da função de
perda que envolvem essas derivadas continuarão, portanto, muito maiores que os demais, e essa disparidade
terá de ser compensada no ajuste dos pesos.

In [ ]:
def build_reference(y_output, x_max=700.0, t_max=1.0):
  L_ref = x_max
  T_ref = t_max
  C_ref = L_ref / T_ref
  P_ref = float(np.abs(y_output).max())

  return L_ref, T_ref, C_ref, P_ref

def normalize_dataset(X_input, y_output, L_ref, T_ref, P_ref):
  X_hat = X_input.copy().astype(np.float32)
  X_hat[:, 0:2] /= L_ref
  X_hat[:, 2:3] /= T_ref
  y_hat = (y_output / P_ref).astype(np.float32)

  return X_hat, y_hat

L_REF, T_REF, C_REF, P_REF = build_reference(y_train)
X_hat, y_hat = normalize_dataset(X_train, y_train, L_REF, T_REF, P_REF)

print(f"L_ref = {L_REF:.1f} m | T_ref = {T_REF:.1f} s | C_ref = {C_REF:.1f} m/s | P_ref = {P_REF:.2f}")
print(f"Entradas x, z : [{X_hat[:,0].min():.3f}, {X_hat[:,0].max():.3f}]")
print(f"Entrada t     : [{X_hat[:,2].min():.3f}, {X_hat[:,2].max():.3f}]")
print(f"Saída p       : [{y_hat.min():.3f}, {y_hat.max():.3f}]")
print(f"Velocidade adimensional: [{1500/C_REF:.2f}, {4500/C_REF:.2f}]")

L_ref = 700.0 m | T_ref = 1.0 s | C_ref = 700.0 m/s | P_ref = 39.40
Entradas x, z : [0.000, 1.000]
Entrada t     : [0.000, 1.000]
Saída p       : [-0.527, 1.000]
Velocidade adimensional: [2.14, 6.43]


### Modelagem da Física: A Equação da Onda e o Termo de Fonte

Nesta secção, definimos o resíduo da Equação da Onda Acústica 2D, que dita o comportamento físico da Inversão Sísmica (FWI).

A equação diferencial parcial adotada para a modelação acústica em meios isotrópicos e de densidade constante é:

$$\left( \frac{\partial^2 p}{\partial x^2} + \frac{\partial^2 p}{\partial z^2} \right) - \frac{1}{c^2(x,z)} \frac{\partial^2 p}{\partial t^2} = s(x, z, t)$$

Onde:
* $p$ é o campo de pressão acústica.
* $c(x,z)$ é o modelo de velocidade de propagação do subsolo.
* $s(x, z, t)$ é o termo de injeção da fonte sísmica.

#### Formulação do Termo de Fonte ($s$)
Para que a PINN aproxime a perturbação real sem sofrer instabilidade de gradiente devido a singularidades (como a função Delta de Dirac), o termo de fonte é aproximado por separação de variáveis:

$$s(x, z, t) = R(t) \cdot G(x, z)$$

**1. Domínio do Tempo (Ricker Wavelet):**
A assinatura temporal do pulso é a segunda derivada de uma função Gaussiana. A frequência dominante $f_0$ é fixada em $15$ Hz, conforme a configuração oficial da família *Vel* do OpenFWI.
$$R(t) = \left( 1 - 2\pi^2 f_0^2 (t - t_0)^2 \right) e^{-\pi^2 f_0^2 (t - t_0)^2}$$
* **O papel do atraso temporal ($t_0$):** A simulação inicia em um estado de repouso absoluto ($t=0$). Se o pulso de Ricker não for deslocado no tempo ($t_0 = 0$), o seu pico de energia máxima ocorrerá exatamente no instante inicial, implicando que metade da onda existiria em tempos negativos. Ao aplicar um atraso temporal $t_0$ (ex: $0.1$ s), a curva inteira é deslocada para o futuro, garantindo que a energia cresce suavemente a partir do zero.

**2. Domínio do Espaço (Aproximação Gaussiana):**
A fonte pontual localizada em $(x_s, z_s)$ é suavizada por uma curva de sino para permitir a diferenciação contínua da rede neural.
$$G(x, z) = e^{-\frac{(x - x_s)^2 + (z - z_s)^2}{\sigma^2}}$$
Onde $\sigma$ é um escalar que controla a dispersão espacial da energia ao redor do epicentro.
* **O papel da dispersão espacial ($\sigma$):** Se for excessivamente pequeno, a curva se aproxima de uma singularidade, inviabilizando a otimização da PINN via Diferenciação Automática devido a derivadas extremas. Se for excessivamente grande, o disparo perde a sua natureza pontual e passa a simular uma onda plana, o que distorce a frente de onda esférica esperada nos sismogramas.

**3. Amplitude ($A$):**
A magnitude do termo de fonte determina a energia injetada no meio e, por consequência, a amplitude
do campo resultante. Ela depende da escala de normalização adotada para a pressão e do fator de
conversão da fonte para as unidades adimensionais, grandezas que não são conhecidas de antemão.

Em vez de arbitrar um valor, a amplitude é tratada como um parâmetro treinável (`dde.Variable`),
otimizado junto com os pesos da rede. O ajuste é guiado pela perda dos dados observados: se o campo
predito for sistematicamente mais fraco que o sismograma medido, o gradiente aumenta $A$. O valor
inicial é estimado por análise de escala, para um campo de amplitude unitária, os termos do operador
da equação da onda ficam na ordem de $\hat{k}^2 \approx 5 \times 10^2$, o que sugere uma fonte de
magnitude comparável.

**Referências:**
1. Deng, C. et al. (2022). *OPENFWI: Large-scale Multi-structural Benchmark Datasets for Full Waveform Inversion*.
2. Virieux, J., & Operto, S. (2009). *An overview of full-waveform inversion in exploration geophysics*.
3. Raissi, M., Perdikaris, P., & Karniadakis, G. E. (2019). *Physics-informed neural networks: A deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations*.
4. Moseley, B., Markham, A., & Nissen-Meyer, T. (2020). *Solving the wave equation with physics-informed deep learning*.


In [ ]:
CFG = {
    "f0": 15.0,          # Hz
    "t0": 0.1,           # s
    "x_fonte": 350.0,    # m
    "z_fonte": 0.0,      # m
    "sigma": 25.0,       # m
    "c": 3000.0,         # m/s (provisório)
    "amp_fonte": 1.0,    # amplitude da fonte (a calibrar)
}

def to_nondim(cfg, L_ref, T_ref, C_ref):
  return {
      "f0": cfg["f0"] * T_ref,
      "t0": cfg["t0"] / T_ref,
      "x_fonte": cfg["x_fonte"] / L_ref,
      "z_fonte": cfg["z_fonte"] / L_ref,
      "sigma": cfg["sigma"] / L_ref,
      "c": cfg["c"] / C_ref,
      "amp_fonte": cfg["amp_fonte"],
  }

CFG_HAT = to_nondim(CFG, L_REF, T_REF, C_REF)

AMPLITUDE = dde.Variable(100.0)

def source_term(X, cfg):
  x, z, t = X[:, 0:1], X[:, 1:2], X[:, 2:3]

  arg = torch.square(np.pi * cfg["f0"] * (t - cfg["t0"]))
  ricker = (1 - 2*arg) * torch.exp(-arg)
  gaussiana = torch.exp(
      -(torch.square(x - cfg["x_fonte"]) + torch.square(z - cfg["z_fonte"])) / cfg["sigma"]**2
  )

  return AMPLITUDE * ricker * gaussiana

def pde_wave_acoustic(X, y):
  dp_xx = dde.grad.hessian(y, X, i=0, j=0)
  dp_zz = dde.grad.hessian(y, X, i=1, j=1)
  dp_tt = dde.grad.hessian(y, X, i=2, j=2)

  c = CFG_HAT["c"]

  return (dp_xx + dp_zz) - (1/c**2)*dp_tt - source_term(X, CFG_HAT)

print("Parâmetros adimensionais:", {k: round(v, 4) for k, v in CFG_HAT.items()})

Parâmetros adimensionais: {'f0': 15.0, 't0': 0.1, 'x_fonte': 0.5, 'z_fonte': 0.0, 'sigma': 0.0357, 'c': 4.2857, 'amp_fonte': 1.0}


### Definição do Domínio Computacional

A função de resíduo definida acima precisa saber onde a física será avaliada. Esse é o papel da
geometria: ela sorteia os pontos de colocação, as coordenadas $(x, z, t)$ nas quais o resíduo da
PDE entra na função de perda.

O domínio é montado em três partes:
* `Rectangle`: o espaço $(x, z) \in [0, 700] \times [0, 700]$ m, correspondente à malha de
  $70 \times 70$ células de $10$ m do FlatVel-A.
* `TimeDomain`: a janela de gravação $t \in [0, 1]$ s.
* `GeometryXTime`: o produto dos dois, formando o domínio espaço-temporal de onde os pontos são
  amostrados.

Diferentemente do método de Diferenças Finitas usado para gerar o dataset, a PINN não trabalha sobre
uma malha estruturada: os pontos são sorteados aleatoriamente no domínio contínuo, sem passo de tempo
fixo e sem a restrição de estabilidade CFL. Em contrapartida, a precisão depende de haver amostras
suficientes onde o campo oscila. Com $c_{min} = 1500$ m/s e $f_0 = 15$ Hz , o
comprimento de onda dominante é $\lambda_0 = c_{min}/f_0 = 100$ m, ou seja, $7$ comprimentos de onda
ao longo do domínio e $15$ períodos na janela de gravação. Esses valores servirão de referência para
dimensionar a quantidade de pontos de colocação.

Note ainda que o domínio aqui definido cobre apenas a região de interesse. A simulação original
utilizou uma moldura adicional de $120$ células de amortecimento em cada borda, que não é replicada: na PINN, ampliar o domínio diluiria os pontos de colocação em um
volume muito maior. O tratamento das bordas é discutido na seção seguinte.

In [ ]:
def build_domain(x_max=1.0, z_max=1.0, t_max=1.0):
  geom = dde.geometry.Rectangle([0.0, 0.0], [x_max, z_max])
  timedomain = dde.geometry.TimeDomain(0.0, t_max)

  return dde.geometry.GeometryXTime(geom, timedomain)

geomtime = build_domain()

pontos = geomtime.random_points(5)
print("Domínio espacial (m):", geomtime.geometry.bbox)
print("Domínio temporal (s):", geomtime.timedomain.t0, "->", geomtime.timedomain.t1)
print("Exemplo de pontos [x, z, t]:\n", pontos)

Domínio espacial (m): (array([0., 0.], dtype=float32), array([1., 1.], dtype=float32))
Domínio temporal (s): 0.0 -> 1.0
Exemplo de pontos [x, z, t]:
 [[0.7231351  0.49676907 0.68645036]
 [0.06184814 0.7892222  0.91668737]
 [0.98135585 0.53343844 0.37171885]
 [0.17948917 0.09314794 0.21258216]
 [0.617128   0.47727764 0.09820291]]


### Amostragem Dirigida da Região da Fonte

Os pontos de colocação são sorteados uniformemente no domínio, o que distribui o esforço computacional
de maneira homogênea. O termo de fonte, entretanto, não é homogêneo: ele é uma gaussiana estreita
centrada em $(x_s, z_s)$, com $\sigma = 25$ m em um domínio de $700 \times 700$ m, ativa apenas durante
a passagem do pulso de Ricker.

A fração do volume espaço-temporal em que a fonte é significativa é, portanto, muito pequena. Sob
amostragem uniforme, pouquíssimos pontos de colocação caem nessa região, e o termo de fonte
praticamente não contribui para a perda da PDE. Como a função de perda é um erro quadrático médio,
alguns poucos pontos entre milhares têm influência desprezível sobre o gradiente, a rede fica livre
para ignorar a fonte, e a equação da onda homogênea admite a solução nula.

A ideia é fornecer explicitamente pontos naquela região. O parâmetro `anchors` do `TimePDE` aceita
um conjunto de coordenadas que são sempre incluídas no treinamento, somando-se às amostradas
aleatoriamente. Geramos essas âncoras cobrindo alguns desvios-padrão em torno do epicentro e alguns
períodos em torno do instante de pico do pulso.

Trata-se de uma forma de amostragem por importância: em vez de distribuir os pontos uniformemente,
concentra-se parte deles onde a física apresenta estrutura. A mesma ideia, aplicada de forma automática
e iterativa a partir do resíduo, é a base dos métodos de amostragem adaptativa (Wu et al., 2023).

**Referências:**

5. Wu, C. et al. (2023). *A comprehensive study of non-adaptive and residual-based adaptive sampling for physics-informed neural networks*.

In [ ]:
def source_anchors(cfg, n=3000, n_sigma=2.5, n_periodos=3.0, seed=42):
  rng = np.random.default_rng(seed)

  raio = n_sigma * cfg["sigma"]
  dur = n_periodos / (np.pi * cfg["f0"])

  x = rng.uniform(cfg["x_fonte"] - raio, cfg["x_fonte"] + raio, n)
  z = rng.uniform(cfg["z_fonte"] - raio, cfg["z_fonte"] + raio, n)
  t = rng.uniform(cfg["t0"] - dur, cfg["t0"] + dur, n)

  pts = np.stack([x, z, t], axis=1)

  return np.clip(pts, 0.0, 1.0).astype(np.float32)

anchors = source_anchors(CFG_HAT)

print("Âncoras geradas:", anchors.shape)
print(f"Cobertura espacial: {2.5*CFG_HAT['sigma']*L_REF:.0f} m em torno da fonte")
print(f"Cobertura temporal: {3.0/(np.pi*CFG['f0'])*1000:.0f} ms em torno de t0")

Âncoras geradas: (3000, 3)
Cobertura espacial: 62 m em torno da fonte
Cobertura temporal: 64 ms em torno de t0


### Condições de Contorno Absorventes

O domínio definido acima é finito, mas o subsolo real não é: a onda que atinge a borda deveria
simplesmente seguir adiante. Sem nenhum tratamento, a rede é livre para produzir reflexões nas quatro
faces, eventos que o sismograma observado não contém, e que fariam a perda da PDE competir com a
perda dos dados.

A condição absorvente de primeira ordem (Clayton & Engquist, 1977) parte da observação de que uma onda
plana que se afasta pela borda direita tem a forma $p = f(x - ct)$. Derivando:

$$\frac{\partial p}{\partial t} = -c f', \qquad \frac{\partial p}{\partial x} = f'
\quad\Longrightarrow\quad \frac{\partial p}{\partial t} + c \frac{\partial p}{\partial x} = 0$$

Generalizando para uma face qualquer, com $\hat{n}$ a normal apontando para fora do domínio:

$$\frac{\partial p}{\partial t} + c \frac{\partial p}{\partial n} = 0$$

O sinal do termo espacial acompanha a orientação da normal: negativo nas faces $x = 0$ e $z = 0$
(normais $-\hat{x}$ e $-\hat{z}$), positivo nas faces $x = 700$ e $z = 700$. Como o dataset foi gerado
com `isFS = false`, não há superfície livre, e a condição é aplicada nas quatro faces, inclusive
no topo.

No DeepXDE, cada condição é um `OperatorBC`: um operador diferencial que deve resultar em zero,
avaliado apenas nos pontos que satisfazem um filtro de fronteira. Diferentemente da moldura de
amortecimento do FDM, que absorve a onda por atenuação gradual, aqui a absorção é imposta como mais um
termo da função de perda, ou seja, é satisfeita apenas de forma aproximada, na medida em que o
treinamento convergir.

Vale registrar a limitação do esquema de primeira ordem: ele é exato somente para incidência normal à
borda. Ondas que chegam obliquamente são absorvidas de forma incompleta e geram reflexões residuais,
tanto mais intensas quanto maior o ângulo de incidência.

**Referências:**
6. Clayton, R., & Engquist, B. (1977). *Absorbing boundary conditions for acoustic and elastic wave equations*.

In [ ]:
def make_abc(eixo, sinal, c):
  def abc(X, y, _):
    dp_n = dde.grad.jacobian(y, X, i=0, j=eixo)   # derivada na direção da normal
    dp_t = dde.grad.jacobian(y, X, i=0, j=2)      # derivada temporal

    return dp_t + sinal * c * dp_n

  return abc

def on_face(eixo, valor):
  return lambda X, on_boundary: on_boundary and np.isclose(X[eixo], valor)

def build_abcs(geomtime, c, x_max=1.0, z_max=1.0):
  faces = [
      (0, -1.0, 0.0),     # esquerda: x = 0,   normal -x
      (0, +1.0, x_max),   # direita:  x = 700, normal +x
      (1, -1.0, 0.0),     # topo:     z = 0,   normal -z
      (1, +1.0, z_max),   # base:     z = 700, normal +z
  ]

  return [
      dde.icbc.OperatorBC(geomtime, make_abc(eixo, sinal, c), on_face(eixo, valor))
      for eixo, sinal, valor in faces
  ]

bcs_absorventes = build_abcs(geomtime, c=CFG_HAT["c"])
print(f"Condições absorventes criadas: {len(bcs_absorventes)}")

Condições absorventes criadas: 4


### Condições Iniciais de Repouso

A simulação parte de um subsolo em repouso absoluto: antes do disparo, não há perturbação alguma no
meio. Isso corresponde a duas restrições em $t = 0$:

$$p(x, z, 0) = 0 \qquad \text{e} \qquad \frac{\partial p}{\partial t}(x, z, 0) = 0$$

A primeira impõe pressão nula; a segunda impõe estado de movimento do meio nulo. Ambas são necessárias:
a equação da onda é de segunda ordem no tempo, e uma equação de segunda ordem exige duas condições
iniciais para ter solução única. Impor apenas $p = 0$ permitiria que a rede iniciasse o campo com
uma "velocidade" arbitrária, produzindo uma onda que não corresponde ao disparo da fonte.

É por essa razão que o atraso temporal $t_0$ foi introduzido no pulso de Ricker. Sem ele, o pico de
energia da fonte ocorreria exatamente em $t = 0$, contradizendo diretamente as condições de repouso.
Com $t_0 = 0{,}1$ s e $f_0 = 15$ Hz, a amplitude do pulso em $t = 0$ é da ordem de $10^{-8}$,
numericamente desprezível frente ao seu valor de pico.

No DeepXDE, a condição sobre $p$ é um `IC` e a condição sobre $\partial p/\partial t$ é um
`OperatorBC` restrito ao instante inicial. Ambas são adicionadas como termos da função de perda,
sendo satisfeitas de forma aproximada conforme o treinamento avança.

In [ ]:
def zero(X):
  return np.zeros((len(X), 1))

def dp_dt_inicial(X, y, _):
  return dde.grad.jacobian(y, X, i=0, j=2)

def build_ics(geomtime):
  ic_pressao = dde.icbc.IC(geomtime, zero, lambda _, on_initial: on_initial)
  ic_velocidade = dde.icbc.OperatorBC(
      geomtime, dp_dt_inicial, lambda X, _: np.isclose(X[2], 0.0)
  )

  return [ic_pressao, ic_velocidade]

ics_repouso = build_ics(geomtime)
print(f"Condições iniciais criadas: {len(ics_repouso)}")

Condições iniciais criadas: 2


### Acoplamento dos Dados Observados

As restrições definidas até aqui, resíduo da PDE, absorção nas bordas e repouso inicial, descrevem
apenas o problema direto: como a onda se propaga dado um modelo de velocidade. Nenhuma delas
contém informação sobre o subsolo específico que gerou o sismograma do FlatVel-A.

O que transforma este problema em uma inversão é o acoplamento dos dados medidos. Impor que a
rede reproduza a pressão registrada pelos receptores é a única restrição que carrega informação sobre
a estrutura real do meio:

$$p(x_r, z_r, t) = p_{obs}(x_r, t)$$

para cada receptor $r$ e cada instante gravado. No DeepXDE isso é feito com `PointSetBC`, que recebe
diretamente as coordenadas e os valores medidos, sem sorteio aleatório: os pontos são exatamente
aqueles onde há medição. Reutilizamos aqui o conjunto tabular $[x, z, t] \rightarrow [p]$ já montado
na etapa de pré-processamento.

Com isso, a função de perda total soma oito termos: o resíduo da PDE, as quatro condições absorventes,
as duas condições iniciais e o ajuste aos dados. Cada termo é o erro quadrático médio do respectivo
resíduo sobre seu conjunto de pontos, ponderado por um peso definido na compilação do modelo.

#### Nota para a fase de otimização

O conjunto observado contém $70000$ pontos (70 receptores $\times$ 1000 instantes), e o volume de
pontos de treino é um dos principais determinantes do consumo de memória da GPU. O teorema da
amostragem (Shannon, 1949) estabelece que reconstruir um sinal exige amostragem de pelo menos o dobro
de sua frequência máxima, o que sugere que boa parte dos $1000$ Hz gravados pode ser redundante para
um pulso de $15$ Hz.

Isso abre a possibilidade de um experimento com previsão analítica na etapa de otimização: variar o
fator de subamostragem temporal e verificar se a degradação da acurácia começa, como previsto, no
ponto em que a taxa efetiva cruza o limite de Nyquist. O conteúdo espectral efetivo do pulso de Ricker
deve ser medido numericamente antes, e não estimado.

**Referências:**

7. Shannon, C. E. (1949). *Communication in the Presence of Noise*. Proceedings of the IRE.

In [ ]:
def build_data_bc(X_input, y_output):
  return dde.icbc.PointSetBC(X_input, y_output, component=0)

bc_dados = build_data_bc(X_hat, y_hat)

print("Pontos observados:", X_hat.shape[0])
print(f"Amplitude observada: [{y_hat.min():.2f}, {y_hat.max():.2f}]")

Pontos observados: 70000
Amplitude observada: [-0.53, 1.00]


### Arquitetura da Rede e Montagem do Modelo

A rede neural é o aproximador do campo de pressão: ela mapeia uma coordenada $(x, z, t)$ no valor
escalar $p$ naquele ponto. Adotamos um Perceptron Multicamadas (MLP) totalmente conectado, arquitetura
padrão da literatura de PINNs (Raissi et al., 2019).

Três escolhas definem a rede:
* **Topologia** `[3] + [128]*6 + [1]`: três entradas, seis camadas ocultas de 128 neurônios e uma
  saída. A profundidade é necessária porque a rede precisa representar as oscilações do campo ao longo
  de toda a janela temporal.
* **Ativação `tanh`**: O resíduo da PDE envolve derivadas de
  segunda ordem, e ativações como a ReLU têm segunda derivada nula em quase todo o domínio, o que
  anularia o termo físico da perda. A `tanh` é suave e infinitamente diferenciável.
* **Inicialização de Glorot**: mantém a variância dos sinais estável ao longo das camadas, evitando
  saturação prematura da `tanh`.

O objeto `TimePDE` reúne a geometria, a função de resíduo e a lista de condições, sendo responsável
por sortear os pontos de colocação (`num_domain`), de fronteira espacial (`num_boundary`) e do
instante inicial (`num_initial`).

Na compilação, `loss_weights` define o peso de cada um dos oito termos da perda, na ordem em que são
montados: primeiro o resíduo da PDE, depois as condições na ordem da lista. O ajuste desses pesos é
empírico e constitui um dos gargalos de treinamento: um termo cujo gradiente domine os demais faz a rede satisfazer uma restrição e negligenciar as
outras.

In [ ]:
def build_model(geomtime, pde, conditions, weights, anchors=None,
                layers=[3] + [128]*6 + [1], lr=1.0e-3,
                num_domain=15000, num_boundary=3000, num_initial=3000,
                external_variables=None):
  data = dde.data.TimePDE(
      geomtime, pde, conditions,
      num_domain=num_domain,
      num_boundary=num_boundary,
      num_initial=num_initial,
      anchors=anchors,
  )

  net = dde.nn.FNN(layers, "tanh", "Glorot normal")
  model = dde.Model(data, net)
  model.compile("adam", lr=lr, loss_weights=weights,
                external_trainable_variables=external_variables)

  return model, data

conditions = bcs_absorventes + ics_repouso + [bc_dados]
weights = [1.0] + [1.0]*4 + [10.0]*2 + [100.0]

model, data = build_model(geomtime, pde_wave_acoustic, conditions, weights, anchors=anchors, external_variables=[AMPLITUDE])

print("Termos da perda:", len(conditions) + 1)
print("Pontos de treino:", len(data.train_x))
print("Parâmetros da rede:", sum(p.numel() for p in model.net.parameters()))

print("train_x_all (domínio+fronteira+inicial):", data.train_x_all.shape)
print("train_x_bc  (pontos das condições):", data.train_x_bc.shape)
print("train_x     (total):", data.train_x.shape)
print("num_bcs (pontos por condição):", data.num_bcs)

Compiling model...
'compile' took 0.000316 s

Termos da perda: 8
Pontos de treino: 104534
Parâmetros da rede: 83201
train_x_all (domínio+fronteira+inicial): (24000, 3)
train_x_bc  (pontos das condições): (80534, 3)
train_x     (total): (104534, 3)
num_bcs (pontos por condição): [749, 751, 2283, 751, 3000, 3000, 70000]


### Treinamento do Modelo

O treinamento é conduzido em duas fases, estratégia usual para PINNs:

**Fase 1 — Adam.** Otimizador de primeira ordem baseado em gradiente estocástico com momento
adaptativo. É robusto a inicializações ruins e conduz os pesos a uma região razoável do espaço de
soluções, mas converge lentamente na vizinhança do mínimo.

**Fase 2 — L-BFGS.** Método quase-Newton que aproxima a matriz Hessiana da função de perda a partir
do histórico de gradientes, permitindo passos muito mais informados. Refina a solução com ordens de
grandeza menos iterações, mas exige um ponto de partida já próximo do mínimo, aplicado desde o
início, tende a estagnar em um mínimo local. Por isso é executado apenas após o Adam.

O `ModelCheckpoint` salva periodicamente os pesos no Google Drive, tornando o treinamento resiliente
à desconexão da sessão do Colab. O objeto `losshistory` retornado registra a evolução de cada um dos
oito termos da perda, permitindo identificar quais restrições convergem e quais estagnam.

In [ ]:
def train_model(model, base_path, iters_adam=1000, iters_lbfgs=5000,
                weights=None, display_every=1000):
  checkpoint = dde.callbacks.ModelCheckpoint(
      f"{base_path}/models/pinn_baseline", save_better_only=True, period=2000
  )

  loss_adam, _ = model.train(
      iterations=iters_adam, display_every=display_every, callbacks=[checkpoint]
  )

  dde.optimizers.set_LBFGS_options(maxiter=iters_lbfgs)
  model.compile("L-BFGS", loss_weights=weights)
  loss_lbfgs, _ = model.train(display_every=display_every)

  return loss_adam, loss_lbfgs

base_path = '/content/drive/MyDrive/tcc_pinn_fwi'

inicio = time.time()
loss_adam, loss_lbfgs = train_model(model, base_path, weights=weights)
print(f"Tempo total: {(time.time() - inicio)/60:.1f} min")

model.save(f"{base_path}/models/pinn_baseline_final")

Training model...

Step      Train loss                                                                          Test loss                                                                           Test metric
0         [2.96e+01, 2.05e-01, 1.03e-01, 1.73e-02, 5.28e-02, 1.64e-02, 1.99e-02, 3.16e-01]    [2.96e+01, 2.05e-01, 1.03e-01, 1.73e-02, 5.28e-02, 1.64e-02, 1.99e-02, 3.16e-01]    []  
1000      [2.90e+01, 7.96e-05, 8.79e-05, 6.03e-05, 8.45e-05, 2.22e-04, 2.91e-05, 1.46e-01]    [2.90e+01, 7.96e-05, 8.79e-05, 6.03e-05, 8.45e-05, 2.22e-04, 2.91e-05, 1.46e-01]    []  

Best model at step 1000:
  train loss: 2.91e+01
  test loss: 2.91e+01
  test metric: []

'train' took 386.222448 s

Compiling model...
'compile' took 0.000451 s

Training model...

Step      Train loss                                                                          Test loss                                                                           Test metric
1000      [2.90e+01, 7.96e-05, 8.79e-05, 6.03e-05, 8.

'/content/drive/MyDrive/tcc_pinn_fwi/models/pinn_baseline_final-1003.pt'

In [ ]:
def diagnosticar_saturacao(model, data, n=5000):
  device = next(model.net.parameters()).device
  X = torch.tensor(data.train_x[:n], dtype=torch.float32, device=device)
  ativacao = torch.tanh(model.net.linears[0](X))
  pct = (ativacao.abs() > 0.99).float().mean().item() * 100
  print(f"Neurônios saturados na 1a camada: {pct:.1f}%")

def diagnosticar_fonte(data, cfg):
  X = data.train_x_all   # pontos que efetivamente entram na perda da PDE

  G = np.exp(
      -((X[:,0] - cfg["x_fonte"])**2 + (X[:,1] - cfg["z_fonte"])**2) / cfg["sigma"]**2
  )
  arg = (np.pi * cfg["f0"] * (X[:,2] - cfg["t0"]))**2
  R = (1 - 2*arg) * np.exp(-arg)
  s = np.abs(cfg["amp_fonte"] * G * R)

  print(f"Pontos com |s| > 0.01: {(s > 0.01).sum()} de {len(X)}")
  print(f"Máximo |s| amostrado : {s.max():.3e}")

def diagnosticar_solucao_trivial(model, X_input, y_output):
  p_pred = model.predict(X_input)

  print(f"max |p| predito  : {np.abs(p_pred).max():.6e}")
  print(f"max |p| observado: {np.abs(y_output).max():.6e}")
  print(f"MSE do modelo    : {np.mean((p_pred - y_output)**2):.6f}")
  print(f"MSE predizendo 0 : {np.mean(y_output**2):.6f}")

diagnosticar_saturacao(model, data)
diagnosticar_fonte(data, CFG_HAT)
diagnosticar_solucao_trivial(model, X_hat, y_hat)

print(f"\nVRAM alocada: {torch.cuda.memory_allocated() / 1e9:.3f} GB")

Neurônios saturados na 1a camada: 0.0%
Pontos com |s| > 0.01: 1412 de 24000
Máximo |s| amostrado : 9.990e-01
max |p| predito  : 1.697358e-03
max |p| observado: 1.000000e+00
MSE do modelo    : 0.001461
MSE predizendo 0 : 0.001461

VRAM alocada: 0.039 GB
